In [1]:
%matplotlib inline
import cv2
import sys
import os

import datacube
import numpy as np
import pandas as pd

sys.path.insert(1, '/home/jovyan/dev/Tools')
from dea_tools.plotting import display_map
from dea_tools.landcover import get_colour_scheme, _get_layer_name, lc_colourmap
from matplotlib import colors as mcolours

from PIL import Image

import matplotlib.pyplot as plt

# Video

In [2]:
# video
def create_video_from_images1(image_folder, output_video):
    # Get list of images
    images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
    images.sort()  # Sort images by name

    # Read the first image to get the dimensions
    frame = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, layers = frame.shape

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_video, fourcc, 30.0, (width, height))

    for i in range(len(images) - 1):
        img1 = cv2.imread(os.path.join(image_folder, images[i]))
        img2 = cv2.imread(os.path.join(image_folder, images[i + 1]))

        # Write the first image
        video.write(img1)

        # Create transition frames
        for alpha in np.linspace(0, 1, num=40):
            blended = cv2.addWeighted(img1, 1 - alpha, img2, alpha, 0)
            video.write(blended)

    # Write the last image
    video.write(cv2.imread(os.path.join(image_folder, images[-1])))

    # Release the video writer
    video.release()


In [3]:

def create_video_from_images2(image_folder, output_video):
    # Get list of images
    images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
    images.sort()  # Sort images by name

    # Read the first image to get the dimensions
    frame = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, layers = frame.shape

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_video, fourcc, 30.0, (width, height))

    for i in range(len(images) - 1):
        img1 = cv2.imread(os.path.join(image_folder, images[i]))
        img2 = cv2.imread(os.path.join(image_folder, images[i + 1]))

        # Extract year from file name
        year1 = images[i].split('_')[-1].split('.')[0]
        year2 = images[i + 1].split('_')[-1].split('.')[0]

        # Add year label to the first image
        font_scale = 3  # Three times larger
        font_thickness = 6  # Make the font bolder
        text_size = cv2.getTextSize(year1, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)[0]
        text_x = width - text_size[0] - 60  # Further from the edge
        text_y = text_size[1] + 100
        cv2.putText(img1, year1, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (105, 105, 105), font_thickness, cv2.LINE_AA)
        video.write(img1)

        # Create transition frames with year label
        for alpha in np.linspace(0, 1, num=60):
            blended = cv2.addWeighted(img1, 1 - alpha, img2, alpha, 0)
            cv2.putText(blended, year2, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (105, 105, 105), font_thickness, cv2.LINE_AA)
            video.write(blended)

    # Add year label to the last image
    img_last = cv2.imread(os.path.join(image_folder, images[-1]))
    year_last = images[-1].split('_')[-1].split('.')[0]
    text_size = cv2.getTextSize(year_last, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)[0]
    text_x = width - text_size[0] - 20  # Further from the edge
    text_y = text_size[1] + 20
    cv2.putText(img_last, year_last, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (105, 105, 105), font_thickness, cv2.LINE_AA)
    video.write(img_last)

    # Release the video writer
    video.release()


In [4]:
# Example usage
#create_video_from_images1('outputs', 'landcover_level4_1km.mp4')

In [5]:


# Folder containing the PNG files
folder_path = 'sml_outputs'

# Get list of files in the folder and sort them
frames = sorted([f for f in os.listdir(folder_path) if f.endswith('.png')])

# Create a list to hold the images
images = []

# Open each frame and append to the images list
for frame in frames:
    img = Image.open(os.path.join(folder_path, frame))
    images.append(img)

# Save the frames as a GIF
images[0].save('sml_landcover_level4_4km.gif', save_all=True, append_images=images[1:], loop=0, duration=400)

print("GIF created successfully and saved as output.gif")

GIF created successfully and saved as output.gif
